In [14]:
import numpy as np
import pandas as pd
from pathlib import Path

from more_itertools import chunked

base_dir = Path('synthesibility_results')
model_paths_dict = {
    # 'SCENT (C+D+P)': 'rgfn_new_filtered/seh/dyn_lib_reward_exploitation_pen_none',
    # 'Graph GA': 'unconstrained/seh/graph_ga',
    'SCENT (C+D+P) smaller': 'rgfn_new_filtered/seh/rgfn_smaller',
    'RGFN smaller': 'rgfn_new_filtered/seh/rgfn_old_smaller',
    # 'RGFN': 'rgfn_new_filtered/seh/rgfn_old',
    # 'REINVENT': 'unconstrained/seh/reinvent',
    # 'FGFN': 'unconstrained/seh/fgfn',
    # 'FGFN (SA)': 'unconstrained/seh/fgfn_sas',
}

In [15]:
from rdkit import DataStructs
from tqdm import tqdm
from typing import List, Any


def get_all_chembl_ecfps():
    from rdkit.DataStructs.cDataStructs import CreateFromBitString
    with open('../data/all_chembl_ecfps.txt') as f:
        fps = [CreateFromBitString(line.strip()) for line in f]
    print(f"Loaded {len(fps)} fps")
    return fps

def compute_diversity(fps: List[Any]) -> List[float]:
    diversities = []
    for i in tqdm(range(len(fps)), desc="diversity"):
        similarities = DataStructs.BulkTanimotoSimilarity(
            fps[i], fps[:i] + fps[i + 1:]
        )
        diversities.append(1 - np.mean(similarities))
    return diversities


def compute_novelty(fps: List[Any], all_chembl_ecfps: List[Any]) -> List[float]:
    novelties = []
    for fp in tqdm(fps, desc="novelty"):
        best_similarity = 0.0
        for batch in chunked(all_chembl_ecfps, 1000):
            similarities = DataStructs.BulkTanimotoSimilarity(fp, batch)
            best_similarity = max(best_similarity, np.max(similarities))
        novelties.append(1 - best_similarity)
    return novelties

def compute_modes(fps: List[Any]) -> List[int]:
    mode_fps = []
    mode_indices = []
    for fp in tqdm(fps, desc="modes"):
        similarities = DataStructs.BulkTanimotoSimilarity(fp, mode_fps)
        if len(similarities) == 0 or max(similarities) < 0.5:
            mode_indices.append(len(mode_fps))
            mode_fps.append(fp)
        else:
            mode_indices.append(similarities.index(max(similarities)))
    return mode_indices

In [16]:
from notebooks.utils.common import _get_fp

all_chembl_ecfps = get_all_chembl_ecfps()
for model, model_path in model_paths_dict.items():
    for seed in range(3):
        df = pd.read_csv(base_dir / model_path / f'sampling/OUT_molecules_{seed}.csv')
        df = df.iloc[:1000]
        novelty_fps = [_get_fp(smiles, "morgan_2", n_bits=1024) for smiles in df['smiles']]
        df['novelty'] = compute_novelty(novelty_fps, all_chembl_ecfps)
        fps = [_get_fp(smiles, "morgan_3") for smiles in df['smiles']]
        df['diversity'] = compute_diversity(fps)
        df.to_csv(base_dir / model_path / f'sampling/OUT_molecules_{seed}_augmented.csv', index=False)

Loaded 1801618 fps


diversity: 100%|██████████| 1000/1000 [00:00<00:00, 5176.47it/s]
